# 03. 모델 학습 및 평가 (Modeling & Evaluation)

전처리된 피처를 사용해 여러 회귀 모델을 학습하고 RMSE / R² 기준으로 비교합니다.  
두 가지 시나리오를 평가합니다.

- **Full Model**: 30K 구간까지의 분할 기록 + 인구통계 + Fatigue Index 포함
- **Early Prediction Model**: 10K 기록까지만 사용 (레이스 초반에 완주 시간 예측)

## 1. 라이브러리 및 데이터 로드

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

from src.preprocessing import load_and_clean, engineer_features, FEATURE_COLS, TARGET_COL

df = load_and_clean('../data/marathon_results_2015.csv')
df = engineer_features(df)
print(f"학습 데이터: {len(df):,}명")

## 2. Train / Test Split

In [ ]:
data = df[FEATURE_COLS + [TARGET_COL]].dropna()
X = data[FEATURE_COLS]
y = data[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train):,}  /  Test: {len(X_test):,}")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

## 3. Full Model 학습 및 평가

In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    r2   = r2_score(y_te, preds)
    print(f"{name:<20} RMSE = {rmse/60:.2f} min ({rmse:.0f}s)   R² = {r2:.4f}")
    return {'RMSE_min': rmse/60, 'R2': r2, 'preds': preds}

full_results = {}
full_results['Ridge']         = evaluate('Ridge',         Ridge(alpha=1.0),  X_train_sc, X_test_sc, y_train, y_test)
full_results['Lasso']         = evaluate('Lasso',         Lasso(alpha=1.0),  X_train_sc, X_test_sc, y_train, y_test)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
full_results['Random Forest'] = evaluate('Random Forest', rf, X_train, X_test, y_train, y_test)

try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
    full_results['XGBoost'] = evaluate('XGBoost', xgb, X_train, X_test, y_train, y_test)
except ImportError:
    print("XGBoost not installed — skip")

try:
    from lightgbm import LGBMRegressor
    lgbm = LGBMRegressor(n_estimators=200, random_state=42, verbose=-1)
    full_results['LightGBM'] = evaluate('LightGBM', lgbm, X_train, X_test, y_train, y_test)
except ImportError:
    print("LightGBM not installed — skip")

## 4. Early Prediction Model (10K까지만 사용)

In [ ]:
EARLY_FEATURES = ['Age', 'Gender_bin', '5K_s', '10K_s', 'Fatigue_Index']

data_e = df[EARLY_FEATURES + [TARGET_COL]].dropna()
Xe = data_e[EARLY_FEATURES]
ye = data_e[TARGET_COL]
Xe_tr, Xe_te, ye_tr, ye_te = train_test_split(Xe, ye, test_size=0.2, random_state=42)

sc_e = StandardScaler()
print("=== Early Prediction (10K 기준) ===")
early_results = {}
early_results['Ridge (Early)'] = evaluate('Ridge (Early)', Ridge(alpha=1.0),
                                            sc_e.fit_transform(Xe_tr), sc_e.transform(Xe_te), ye_tr, ye_te)

rf_e = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
early_results['RF (Early)'] = evaluate('RF (Early)', rf_e, Xe_tr, Xe_te, ye_tr, ye_te)

## 5. 모델 성능 비교 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_res = {**full_results, **early_results}
names  = list(all_res.keys())
rmses  = [v['RMSE_min'] for v in all_res.values()]
r2s    = [v['R2'] for v in all_res.values()]
colors = ['steelblue'] * len(full_results) + ['tomato'] * len(early_results)

axes[0].barh(names, rmses, color=colors)
axes[0].set_xlabel('RMSE (minutes)')
axes[0].set_title('RMSE Comparison (lower is better)')
axes[0].invert_yaxis()

axes[1].barh(names, r2s, color=colors)
axes[1].set_xlabel('R² Score')
axes[1].set_title('R² Comparison (higher is better)')
axes[1].set_xlim(0.96, 1.0)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Random Forest 피처 중요도

In [ ]:
fi = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
fi.plot(kind='barh', color='steelblue')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(fi.sort_values(ascending=False).round(4).to_string())

## 7. 예측값 vs 실제값 (Actual vs Predicted)

In [ ]:
ridge_final = Ridge(alpha=1.0)
ridge_final.fit(X_train_sc, y_train)
preds_ridge = ridge_final.predict(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full model
axes[0].scatter(y_test / 60, preds_ridge / 60, alpha=0.1, s=5, color='steelblue')
mn, mx = y_test.min() / 60, y_test.max() / 60
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual (minutes)')
axes[0].set_ylabel('Predicted (minutes)')
axes[0].set_title('Full Model — Ridge: Actual vs Predicted')
axes[0].legend()

# Residuals
residuals = (y_test - preds_ridge) / 60
axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Residual (minutes)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].axvline(0, color='red', linestyle='--')

plt.tight_layout()
plt.show()